In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from feature_engine.datetime import DatetimeFeatures
from joblib import dump

In [2]:
train_df = pd.read_csv('data/training.csv')
val_df = pd.read_csv('data/validation.csv')
test_df = pd.read_csv('data/testing.csv')

In [3]:
train_df['dataset'] = 'train'
val_df['dataset'] = 'val'
test_df['dataset'] = 'test'
df = pd.concat([train_df, val_df, test_df], axis=0).reset_index(drop=True)

In [4]:
df['Additional_Info'] = df['Additional_Info'].replace('No Info', 'No info')

In [5]:
doj_columns = ['Date_of_Journey']
time_columns = ['Dep_Time', 'Arrival_Time']
categorical_cols = ['Airline', 'Source', 'Destination', 'Additional_Info']
numerical_cols = ['Duration', 'Total_Stops']

In [6]:
train_processed = df[df['dataset'] == 'train'].drop(columns=['dataset'])
val_processed = df[df['dataset'] == 'val'].drop(columns=['dataset'])
test_processed = df[df['dataset'] == 'test'].drop(columns=['dataset'])

In [7]:
X_train = train_processed.drop(columns=['Price'])
y_train = train_processed['Price']

X_val = val_processed.drop(columns=['Price'])
y_val = val_processed['Price']

X_test = test_processed.drop(columns=['Price'])
y_test = test_processed['Price']

In [8]:
doj_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("extractor", DatetimeFeatures(features_to_extract=['week', 'day_of_week', 'month', 'day_of_month'], format='mixed')),
    ("scaler", StandardScaler())
])
time_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("extractor", DatetimeFeatures(features_to_extract=['hour', 'minute'], format='mixed')),
    ("scaler", StandardScaler())
])

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

In [9]:
transformer = ColumnTransformer(
    transformers=[
        ('numerical', numerical_pipeline, numerical_cols),
        ('categorical', categorical_pipeline, categorical_cols),
        ('doj', doj_transformer, doj_columns),
        ('time', time_transformer, time_columns)
    ],
    remainder='passthrough',
)

In [10]:
model = Pipeline([
    ('transformer', transformer),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )),
])
model.fit(X_train, y_train)

Pipeline(steps=[('transformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Duration', 'Total_Stops']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'...
                                                                                                         'day_of_month'],
                                                                                    format='mixed')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Date_of_Journey']),
                                                 ('time',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('extractor',
                                                                   DatetimeFeatures(features_to_extract=['hour',
                                                                                                         'minute'],
                                                                                    format='mixed')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Dep_Time',
                                                   'Arrival_Time'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

In [11]:
val_score = model.score(X_val, y_val) * 100
test_score = model.score(X_test, y_test) * 100

In [12]:
X = pd.concat([X_train, X_val, X_test], axis=0).reset_index(drop=True)
y = pd.concat([y_train, y_val, y_test], axis=0).reset_index(drop=True)

In [13]:
cv_score = np.array(cross_val_score(model, X, y, scoring='r2', cv=10)).mean() * 100

In [14]:
val_score, test_score, cv_score

(88.7375642737794, 92.30912338307678, np.float64(88.42469446383774))

In [15]:
dump(model, "models/predictor.joblib")

['models/predictor.joblib']